In [2]:
import pandas as pd
import json
import os

if os.path.isdir("../data/"):
    os.chdir("../")

from src.file_io import *
from src.utils import get_channel_names_dict

%load_ext autoreload
%autoreload 2

# Channels verification and standardization

### youtube channels, nothing much to do

In [2]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')

### tiktok channels : 
- extract channels from json
- standardize username using the dictionnary `get_channel_names_dict`
- extract orientation using the dictionnary `get_news_channel_orientation` for news and the party for pp 

#### NEWS

In [27]:
with open("data/tiktok/channels/news_channels.json") as f:
    nm_tt_channels = json.load(f)

nm_tt_channels = pd.DataFrame.from_dict(nm_tt_channels['channels'], orient="index").reset_index().rename(columns={
    'index':'username'
})

# name standardization
nm_tt_channels['name_standard'] = nm_tt_channels['username'].map(get_channel_names_dict())
nm_tt_channels['orientation'] = nm_tt_channels['name_standard'].map(get_news_channel_orientation())

In [28]:
set(nm_yt_channels.name_standard).symmetric_difference(set(nm_tt_channels.name_standard))

{'Euronews', 'LCI'}

In [31]:
nm_tt_channels.to_json("data/tiktok/channels/news_channels.json", orient='records', indent=2, force_ascii=False)

#### PP 

In [21]:
with open("data/tiktok/channels/pp_channels.json") as f:
    pp_tt_channels = json.load(f)
    
pp_tt_channels = pd.DataFrame.from_dict(pp_tt_channels['channels'], orient="index").reset_index().rename(columns={
    'index': 'username'
})
pp_tt_channels['name_standard'] = pp_tt_channels['username'].apply(lambda x: get_channel_names_dict().get(x,x))
pp_tt_channels['orientation'] = pp_tt_channels['party'].apply(lambda x: get_party_orientation()[x])

In [22]:
mask = ~pp_tt_channels['username'].isin(get_channel_names_dict().keys())
pp_tt_channels.loc[mask, 'name_standard'] = pp_tt_channels.loc[mask, 'display_name']

In [33]:
pp_tt_channels.to_json("data/tiktok/channels/pp_channels.json", orient='records', indent=2, force_ascii=False)

#### final verif

In [35]:
for df in [nm_tt_channels, pp_tt_channels]:
    for column in df.columns:
        print(f"{column}:\n\t- {sum(df[column].isna())} na values\n\t- {len(df[df[column] == ""])} empty string values")
    print()

username:
	- 0 na values
	- 0 empty string values
bio_description:
	- 0 na values
	- 1 empty string values
display_name:
	- 0 na values
	- 0 empty string values
follower_count:
	- 0 na values
	- 0 empty string values
following_count:
	- 0 na values
	- 0 empty string values
is_verified:
	- 0 na values
	- 0 empty string values
likes_count:
	- 0 na values
	- 0 empty string values
video_count:
	- 0 na values
	- 0 empty string values
name_standard:
	- 0 na values
	- 0 empty string values
orientation:
	- 0 na values
	- 0 empty string values

username:
	- 0 na values
	- 0 empty string values
bio_description:
	- 0 na values
	- 1 empty string values
display_name:
	- 0 na values
	- 0 empty string values
follower_count:
	- 0 na values
	- 0 empty string values
following_count:
	- 0 na values
	- 0 empty string values
is_verified:
	- 0 na values
	- 0 empty string values
likes_count:
	- 0 na values
	- 0 empty string values
video_count:
	- 0 na values
	- 0 empty string values
collected_at:
	- 0 na val

Channels looks ok.

# Videos verification

# Change youtube video csv in jsonl

In [2]:
nm_videos_2022 = pd.read_csv("data/youtube/videos/news_videos_2022.csv")
nm_videos_2022.to_json("data/youtube/videos/news_videos_2022.jsonl",lines=True, orient="records")

In [3]:
nm_videos_2024 = pd.read_csv("data/youtube/videos/news_videos_2024.csv")
nm_videos_2024.to_json("data/youtube/videos/news_videos_2024.jsonl",lines=True, orient="records")

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_39922/1325777972.py:1: DtypeWarning: Columns (0: duration, 1: ORG_orientation) have mixed types. Specify dtype option on import or set low_memory=False.
  nm_videos_2024 = pd.read_csv("data/youtube/videos/news_videos_2024.csv")


In [4]:
pp_videos_2022 = pd.read_csv("data/youtube/videos/pp_videos_2022.csv")
pp_videos_2022.to_json("data/youtube/videos/pp_videos_2022.jsonl",lines=True, orient="records")

In [5]:
pp_videos_2024 = pd.read_csv("data/youtube/videos/pp_videos_2024.csv")
pp_videos_2024.to_json("data/youtube/videos/pp_videos_2024.jsonl",lines=True, orient="records")

### remove 'P0D' from youtube videos

In [93]:
nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl",lines=True)
print(len(nm_videos_2022))
nm_videos_2022 = nm_videos_2022[nm_videos_2022.duration != 'P0D']
print(len(nm_videos_2022))

37481
37481


In [95]:
nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl",lines=True)
print(len(nm_videos_2024))
nm_videos_2024 = nm_videos_2024[nm_videos_2024.duration != 'P0D']
print(len(nm_videos_2024))
nm_videos_2024['duration'] = nm_videos_2024['duration'].apply(float).apply(int)
nm_videos_2024.to_json("data/youtube/videos/news_videos_2024.jsonl",lines=True, orient='records')

40984
40982


# Add transcripts to youtube video jsonl

In [2]:
def clean_transcript(transcripts):
    transcripts = pd.DataFrame.from_dict(transcripts, columns=["transcript"], orient="index").reset_index().rename(
        columns={"index": "videoId"}
    )
    error_messages = ["[video_not_available]", "[video_private]", "[video_age_restricted]", "[live_video]",
                      "[copyright_grounds]", "[copyright_claims]",
                      "[video_removed]", "[suicide_self_harm_topics]", "[video_not_available]",
                      "[video_graphic_or_violent]", "[transcripts_disabled]", "[no_autogenerated_transcript]"]
    transcripts.loc[transcripts['transcript'].isin(error_messages), "error_transcript"] = \
        transcripts.loc[transcripts['transcript'].isin(error_messages), "transcript"]
    transcripts.loc[transcripts['transcript'].isin(error_messages), "transcript"] = ""
    return transcripts

In [7]:
with open("data/youtube/transcripts/news_transcripts_2022.json") as f:
    nm_transcripts_2022 = json.load(f)
len(nm_transcripts_2022)

nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
nm_videos_2022 = nm_videos_2022.merge(clean_transcript(nm_transcripts_2022), on="videoId")
nm_videos_2022.to_json("data/youtube/videos/news_videos_2022.jsonl",lines=True, orient="records")

In [42]:
with open("data/youtube/transcripts/news_transcripts_2024.json") as f:
    nm_transcripts_2024 = json.load(f)
len(nm_transcripts_2024)

nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
nm_videos_2024 = nm_videos_2024.merge(clean_transcript(nm_transcripts_2024), on="videoId")
nm_videos_2024.to_json("data/youtube/videos/news_videos_2024.jsonl",lines=True, orient="records")

In [54]:
with open("data/youtube/transcripts/pp_transcripts_2022.json") as f:
    pp_transcripts_2022 = json.load(f)
len(pp_transcripts_2022)

pp_videos_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl", lines=True)
len_videos = len(pp_videos_2022)
pp_videos_2022 = pp_videos_2022.merge(clean_transcript(pp_transcripts_2022), on="videoId")
assert len_videos == len(pp_videos_2022)
pp_videos_2022.to_json("data/youtube/videos/pp_videos_2022.jsonl",lines=True, orient="records")

In [53]:
with open("data/youtube/transcripts/pp_transcripts_2024.json") as f:
    pp_transcripts_2024 = json.load(f)
len(pp_transcripts_2024)

pp_videos_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl", lines=True)
len_videos = len(pp_videos_2024)
pp_videos_2024 = pp_videos_2024.merge(clean_transcript(pp_transcripts_2024), on="videoId")
assert len_videos == len(pp_videos_2024)
pp_videos_2024.to_json("data/youtube/videos/pp_videos_2024.jsonl",lines=True, orient="records")

# Put TikTok in jsonl as well

In [5]:
with open('data/tiktok/videos/news_videos_2022.json') as f:
    tt_nm_videos_2022 = json.load(f)
df = pd.DataFrame.from_records(tt_nm_videos_2022)
df.to_json('data/tiktok/videos/news_videos_2022.jsonl', lines=True, orient='records')

In [12]:
with open('data/tiktok/videos/news_videos_2024.json') as f:
    tt_nm_videos_2024 = json.load(f)
df = pd.DataFrame.from_records(tt_nm_videos_2024)
df.to_json('data/tiktok/videos/news_videos_2024.jsonl', lines=True, orient='records')

In [13]:
with open('data/tiktok/videos/pp_videos_2022.json') as f:
    tt_pp_videos_2022 = json.load(f)
df = pd.DataFrame.from_records(tt_pp_videos_2022)
df.to_json('data/tiktok/videos/pp_videos_2022.jsonl', lines=True, orient='records')

In [ ]:
with open('data/tiktok/videos/pp_videos_2024.json') as f:
    tt_pp_videos_2024 = json.load(f)
df = pd.DataFrame.from_records(tt_pp_videos_2024)
df.to_json('data/tiktok/videos/pp_videos_2024.jsonl', lines=True, orient='records')

# add missing transcripts

In [3]:
def load_transcript(videoid, platform):
    try:
        with open(f'data/{platform}/videos/transcripts/{videoid}.json') as f:
            transcript = json.load(f)
        return transcript['text']
    except FileNotFoundError as e:
        return None

In [3]:
news_videos_2022, 5824 missing transcripts
news_videos_2022, 2219 missing transcripts
news_videos_2024, 9510 missing transcripts
news_videos_2024, 7880 missing transcripts
pp_videos_2022, 5 missing transcripts
pp_videos_2022, 5 missing transcripts
pp_videos_2024, 12 missing transcripts
pp_videos_2024, 12 missing transcripts

SyntaxError: invalid syntax (753161706.py, line 1)

In [4]:
platform = 'youtube'
for channel_type in ['news', 'pp']:
    for year in ['2022', '2024']:
        path = f'data/{platform}/videos/{channel_type}_videos_{year}.jsonl'
        df = pd.read_json(path, lines=True)
        len_df = len(df)

        if platform == 'youtube':
            transcript_loc = 'transcript'
            id_loc = 'videoId'
        elif platform == 'tiktok':
            transcript_loc = 'voice_to_text'
            id_loc = 'id'

        mask = (df[transcript_loc].isna()) | (df[transcript_loc] == '')
        print(f"{channel_type}_videos_{year}, {len(df[mask])} missing transcripts")

        df.loc[mask, transcript_loc] = df.loc[mask, id_loc].apply(load_transcript, args=(platform,))

        assert len(df) == len_df

        # split off unavailable videos
        unavailable_mask = df[transcript_loc] == '[video unavailable]'
        df_unavailable = df[unavailable_mask]
        df = df[~unavailable_mask]

        print(f"{channel_type}_videos_{year}, {len(df_unavailable)} unavailable videos set aside")

        still_missing = (df[transcript_loc].isna()) | (df[transcript_loc] == '')
        print(f"{channel_type}_videos_{year}, {len(df[still_missing])} missing transcripts")

        df.to_json(path, lines=True, orient='records')
        if len(df_unavailable) > 0:
            df_unavailable.to_json(
                f'data/{platform}/videos/{channel_type}_videos_not_available_{year}.jsonl',
                lines=True, orient='records'
            )

news_videos_2022, 2219 missing transcripts
news_videos_2022, 0 unavailable videos set aside
news_videos_2022, 2219 missing transcripts


news_videos_2024, 2443 missing transcripts


news_videos_2024, 1291 unavailable videos set aside
news_videos_2024, 892 missing transcripts


pp_videos_2022, 5 missing transcripts
pp_videos_2022, 0 unavailable videos set aside
pp_videos_2022, 5 missing transcripts


pp_videos_2024, 12 missing transcripts
pp_videos_2024, 0 unavailable videos set aside
pp_videos_2024, 12 missing transcripts


In [10]:
for platform in ['youtube', 'tiktok']:
    print(platform)
    for channel_type in ['news', 'pp']:
        for year in ['2022', '2024']:
            path_df = f'data/{platform}/videos/{channel_type}_videos_{year}.jsonl'
            path_unavailable_df = f'data/{platform}/videos/{channel_type}_videos_not_available_{year}.jsonl'
            df = pd.read_json(path_df, lines=True)
            try:
                df_unavailable = pd.read_json(path_unavailable_df, lines=True)
                len_unavailable = len(df_unavailable)
            except FileNotFoundError:
                len_unavailable = 0
            
            print(f"{channel_type}_videos_{year} : {len(df)}/{len(df) + len_unavailable} available videos")

youtube


news_videos_2022 : 37481/37481 available videos


news_videos_2024 : 39450/40741 available videos
pp_videos_2022 : 1319/1319 available videos


pp_videos_2024 : 2520/2520 available videos
tiktok
news_videos_2022 : 1760/1760 available videos


news_videos_2024 : 6983/6983 available videos
pp_videos_2022 : 812/812 available videos
pp_videos_2024 : 3960/3960 available videos


In [5]:
test = pd.read_json(f'data/youtube/videos/news_videos_2024.jsonl', lines=True)

In [8]:
videos_test = ['jeNJdb8FCbQ', 'jyZzRmRjRlc', 'kFcVXv4VUuw', 'kIySJ50Rrmg', 'kOe0s8XA4fY', 'lazP5N9cAq0', 'm-iOdKlr-s0', 'm5DpPsptw50', 'mBkGHDwZ58A', 'mlS4U2j0_e0', 'n9tJT_jA0Hk', 'nEd5rWa2LSY', 'oaU2G6W6Oww', 'pk_RfOawqVY', 'q0bNQx58ANs', 'qJHQ-NN41kQ', 'qn45vD8AWFg', 'qxoEU8bMnAI', 'r8u2MLtMedc', 'rfxbbQZ4oAM', 'thrSIfiqEAQ', 'vu-4vRXfmVY', 'wDUDApI09yg', 'wZh6jm-vDCg', 'x1kx8P8Q-_I', 'y8yQPeU4ZIs', 'ytHn409W-_4', 'zQWoLohWM7A', 'zg9BxA3X77c', 'zyrE80jWh1c']

In [9]:
test[test['videoId'].isin(videos_test)]

,videoId,publishedAt,channelId,title,description,channelTitle,playlistId,duration,snippet.tags,snippet.categoryId,...,Jordan Bardella,Rima Hassan,Marine Le Pen,Manon Aubry,Marion Maréchal,Emmanuel Macron,Gabriel Attal,transcript,error_transcript,name_standard
10056,kIySJ50Rrmg,2024-05-02T11:15:03Z,UCzBoVtUqSyAyr0kF5ABW7gQ,"Dr. Jimmy Mohamed : Jardiner, c'est bon pour l...","Jardiner, c'est bon pour la santé ! Chronique ...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,49,RTL#jimmy mohamed#jardinage santé#jardiner santé,26,...,False,False,False,False,False,False,False,Tondre sa pelouse pendant une heure équivaut à...,[transcripts_disabled],RTL
10058,kFcVXv4VUuw,2024-04-26T11:23:13Z,UCzBoVtUqSyAyr0kF5ABW7gQ,"Recrudescence de la coqueluche en Europe, appe...","Recrudescence de la coqueluche en Europe, appe...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,43,RTL#jimmy mohamed#coqueluche,26,...,False,False,False,False,False,False,False,Alors la coqueluche se transmet par les postil...,[transcripts_disabled],RTL
10064,mBkGHDwZ58A,2024-04-12T08:40:00Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Aide à mourir : La possibilité de partir donne...,"""Je crois qu'il faut être un peu empathique et...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,544,RTL,25,...,False,False,False,False,False,False,False,Bonjour Jérôme Caz. Bonjour Yves Calvi. Merci ...,[transcripts_disabled],RTL
10065,mlS4U2j0_e0,2024-04-08T18:26:53Z,UCzBoVtUqSyAyr0kF5ABW7gQ,"Marc-Antoine Le Bret imite Orelsan, François H...",Retrouvez les imitations de Marc-Antoine Le Br...,RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,221,RTL,25,...,False,False,False,False,False,False,False,Avec Marc-Antoine Lebré. C'est terrible. Marc-...,[transcripts_disabled],RTL
10066,jyZzRmRjRlc,2024-04-04T16:15:01Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Cyril Lignac : le risotto printanier,Le risotto printanier : la recette du jour de ...,RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,111,RTL#risotto printanier#cyril lignac#lignac#lig...,26,...,False,False,False,False,False,False,False,Et maintenant un festival d'astuces. Ciao Cyri...,[transcripts_disabled],RTL
10067,qn45vD8AWFg,2024-04-03T14:15:06Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Cyril Lignac : le sablé breton aux fruits roug...,Le sablé breton aux fruits rouges et amandes :...,RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,128,RTL#sablé breton#cyril lignac#lignac#lignac re...,26,...,False,False,False,False,False,False,False,"Alors, c'est le début des fraises à Plougastel...",[transcripts_disabled],RTL
10068,m-iOdKlr-s0,2024-04-01T09:10:00Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Des profs ne lâchent rien : Je vais faire l'en...,"Des profs ne lâchent rien : ""Je vais faire l'e...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,46,RTL#philippe caverivière#caverivière#caveriviè...,23,...,False,False,False,False,False,False,False,"La laïcité recule, on constate que les profs o...",[transcripts_disabled],RTL
10069,qJHQ-NN41kQ,2024-03-29T09:57:29Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Le tutoriel du jour par Élie Semoun,À l’occasion de la sortie du film « Ducobu pas...,RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,16,RTL,25,...,False,False,False,False,False,False,False,On m'a demandé comment il fallait s'énerver co...,[transcripts_disabled],RTL
10070,oaU2G6W6Oww,2024-03-27T18:10:26Z,UCzBoVtUqSyAyr0kF5ABW7gQ,"Gabriel Attal, Philippe Etchebest, Jeff Tuche....","🤣 Gabriel Attal : ""Je rappelle que si les fran...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,199,RTL#marc-antoine le bret#le bret rtl#rtl humour,23,...,False,False,False,False,False,False,True,"C'est parti pour le bré ! Gabriel Attal, vous ...",[transcripts_disabled],RTL
10071,thrSIfiqEAQ,2024-03-25T14:20:38Z,UCzBoVtUqSyAyr0kF5ABW7gQ,Le Grand Jury des Auditeurs de RTL : Raphaël G...,"""Notre offre politique est nouvelle, je n'ai p...",RTL,UUzBoVtUqSyAyr0kF5ABW7gQ,2105,RTL#Actualités#Divertissement#Musique,25,...,False,False,False,False,False,False,False,Et nous disons bonjour très officiellement à R...,[transcripts_disabled],RTL


## after collecting NER, add just the clean NER results to our main jsonl for each year

In [11]:
for year in ['2022', '2024']:
    df_w_NER = pd.read_json(f'data/tiktok/videos/news_videos_{year}_NER.jsonl', lines=True)
    df_main = pd.read_json(f'data/tiktok/videos/news_videos_{year}.jsonl', lines=True)
    df_main = df_main.merge(df_w_NER[['videoId', 'PER']], left_on = 'id', right_on = 'videoId', how='outer').drop(columns=['videoId'])
    df_main.to_json(f'data/tiktok/videos/news_videos_{year}.jsonl', lines=True, orient='records', force_ascii=False)

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_79810/837651053.py:5: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_main.to_json(f'data/tiktok/videos/news_videos_{year}.jsonl', lines=True, orient='records', force_ascii=False)


/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_79810/837651053.py:5: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_main.to_json(f'data/tiktok/videos/news_videos_{year}.jsonl', lines=True, orient='records', force_ascii=False)


### remove photos from tiktok files(post with duration == 0)

In [36]:
for filename in ["data/tiktok/videos/news_videos_2022.jsonl",
                "data/tiktok/videos/news_videos_2024.jsonl",
                "data/tiktok/videos/pp_videos_2022.jsonl",
                "data/tiktok/videos/pp_videos_2024.jsonl"
                ]:
    tt_videos = pd.read_json(filename, lines=True)
        
    print(len(tt_videos))
    
    final_videos = []
    tt_photos = []
    try:
        final_videos = tt_videos[tt_videos['video_duration'] > 0]
        tt_photos = tt_videos[tt_videos['video_duration'] == 0]
    except KeyError:
        final_videos = tt_videos[tt_videos['duration'] > 0]
        tt_photos = tt_videos[tt_videos['duration'] == 0]
    
    print(len(final_videos))
    print(len(tt_photos))
    
    tt_photos.to_json(filename.replace('video', 'photo'), lines=True, orient='records', force_ascii=False)
    final_videos.to_json(filename, lines=True, orient='records', force_ascii=False)    

1832
1832
0
6983
6983
0


/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:22: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tt_photos.to_json(filename.replace('video', 'photo'), lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:23: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  final_videos.to_json(filename, lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:22: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tt_photos.to_json(filename.replace('video', 'photo'), lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7l

812
812
0
3960
3960
0


/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:22: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tt_photos.to_json(filename.replace('video', 'photo'), lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:23: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  final_videos.to_json(filename, lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_89816/1657382987.py:22: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tt_photos.to_json(filename.replace('video', 'photo'), lines=True, orient='records', force_ascii=False)
/var/folders/s8/86mtzm812zdfvh1cgg7l

# Standard names in video files 
Take the `name_standard` from the channel files, and merge them so that the standardized name are directly on video files

### youtube

#### NEWS

In [41]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')

In [45]:
nm_yt_videos_2022 = pd.read_json('data/youtube/videos/news_videos_2022.jsonl', lines=True)
nm_yt_videos_2022_standard_name = nm_yt_videos_2022.merge(nm_yt_channels[['channelId', 'name_standard']], on='channelId', how = 'left')
assert len(nm_yt_videos_2022) == len(nm_yt_videos_2022_standard_name)
assert sum(nm_yt_videos_2022_standard_name['name_standard'].isna()) == 0 # assert each row has name_standard value

In [48]:
nm_yt_videos_2022_standard_name.to_json('data/youtube/videos/news_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)

In [49]:
nm_yt_videos_2024 = pd.read_json('data/youtube/videos/news_videos_2024.jsonl', lines=True)
nm_yt_videos_2024_standard_name = nm_yt_videos_2024.merge(nm_yt_channels[['channelId', 'name_standard']], on='channelId', how = 'left')
assert len(nm_yt_videos_2024) == len(nm_yt_videos_2024_standard_name) # assert same number of lines between the original and the new
assert sum(nm_yt_videos_2024_standard_name['name_standard'].isna()) == 0 # assert each row has name_standard value

In [51]:
nm_yt_videos_2024_standard_name.to_json('data/youtube/videos/news_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)

#### PP

In [43]:
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')

In [54]:
pp_yt_videos_2022 = pd.read_json('data/youtube/videos/pp_videos_2022.jsonl', lines=True)
pp_yt_videos_2022_standard_name = pp_yt_videos_2022.merge(pp_yt_channels[['channelId', 'name_standard']], on='channelId', how = 'left')
assert len(pp_yt_videos_2022) == len(pp_yt_videos_2022_standard_name)
assert sum(pp_yt_videos_2022_standard_name['name_standard'].isna()) == 0 # assert each row has name_standard value

In [56]:
pp_yt_videos_2022_standard_name.to_json('data/youtube/videos/pp_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)

In [44]:
pp_yt_videos_2024 = pd.read_json('data/youtube/videos/pp_videos_2024.jsonl', lines=True)
pp_yt_videos_2024_standard_name = pp_yt_videos_2024.merge(pp_yt_channels[['channelId', 'name_standard']], on='channelId', how = 'left')
assert len(pp_yt_videos_2024) == len(pp_yt_videos_2024_standard_name)
assert sum(pp_yt_videos_2024_standard_name['name_standard'].isna()) == 0 # assert each row has name_standard value

In [45]:
pp_yt_videos_2024_standard_name.to_json('data/youtube/videos/pp_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)

### TIKTOK

#### NM

In [12]:
nm_tt_channels = pd.read_json('data/tiktok/channels/news_channels.json')

In [14]:
nm_tt_videos_2022 = pd.read_json('data/tiktok/videos/news_videos_2022.jsonl', lines=True)
nm_tt_videos_2022_standard_name = nm_tt_videos_2022.merge(nm_tt_channels[['username', 'name_standard']], on='username', how = 'left')
assert len(nm_tt_videos_2022) == len(nm_tt_videos_2022_standard_name)
mask = nm_tt_videos_2022_standard_name['name_standard'].isna()
assert sum(mask) == 0, display(nm_tt_videos_2022_standard_name[mask]) # assert each row has name_standard value

In [16]:
nm_tt_videos_2022_standard_name.to_json('data/tiktok/videos/news_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_2623/2313393039.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  nm_tt_videos_2022_standard_name.to_json('data/tiktok/videos/news_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)


In [17]:
nm_tt_videos_2024 = pd.read_json('data/tiktok/videos/news_videos_2024.jsonl', lines=True)
nm_tt_videos_2024_standard_name = nm_tt_videos_2024.merge(nm_tt_channels[['username', 'name_standard']], on='username', how = 'left')
assert len(nm_tt_videos_2024) == len(nm_tt_videos_2024_standard_name)
mask = nm_tt_videos_2024_standard_name['name_standard'].isna()
assert sum(mask) == 0, display(nm_tt_videos_2024_standard_name[mask]) # assert each row has name_standard value

In [19]:
nm_tt_videos_2024_standard_name.to_json('data/tiktok/videos/news_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_2623/3446240473.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  nm_tt_videos_2024_standard_name.to_json('data/tiktok/videos/news_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)


#### PP

In [20]:
pp_tt_channels = pd.read_json('data/tiktok/channels/pp_channels.json')

In [22]:
pp_tt_videos_2022 = pd.read_json('data/tiktok/videos/pp_videos_2022.jsonl', lines=True)
pp_tt_videos_2022_standard_name = pp_tt_videos_2022.merge(pp_tt_channels[['username', 'name_standard']], on='username', how = 'left')
assert len(pp_tt_videos_2022) == len(pp_tt_videos_2022_standard_name)
mask = pp_tt_videos_2022_standard_name['name_standard'].isna()
assert sum(mask) == 0, display(pp_tt_videos_2022_standard_name[mask]) # assert each row has name_standard value

In [24]:
pp_tt_videos_2022_standard_name.to_json('data/tiktok/videos/pp_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_2623/4220655353.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  pp_tt_videos_2022_standard_name.to_json('data/tiktok/videos/pp_videos_2022.jsonl', lines=True, orient='records', force_ascii=False)


In [25]:
pp_tt_videos_2024 = pd.read_json('data/tiktok/videos/pp_videos_2024.jsonl', lines=True)
pp_tt_videos_2024_standard_name = pp_tt_videos_2024.merge(pp_tt_channels[['username', 'name_standard']], on='username', how = 'left')
assert len(pp_tt_videos_2024) == len(pp_tt_videos_2024_standard_name)
mask = pp_tt_videos_2024_standard_name['name_standard'].isna()
assert sum(mask) == 0, display(pp_tt_videos_2024_standard_name[mask]) # assert each row has name_standard value

In [27]:
pp_tt_videos_2024_standard_name.to_json('data/tiktok/videos/pp_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)

/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_2623/2185624133.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  pp_tt_videos_2024_standard_name.to_json('data/tiktok/videos/pp_videos_2024.jsonl', lines=True, orient='records', force_ascii=False)


In [50]:
pd.read_json('data/youtube/videos/pp_videos_2024.jsonl', lines=True)[['name_standard', 'videoId', 'transcript']]

,name_standard,videoId,transcript
0,LO,IQnDgTmunaM,donc je suppose que c'est pas pour vous mettre...
1,LFI,FlInZGSqVLY,je m'appelle Rima Hassan j'ai 31 ans j'ai gran...
2,LFI,Juk_lgl_6hg,à Gaza ce sont des crimes de guerre et selon l...
3,NPA,gyv_4eYRkOU,est un chanteur de Ré jamaïcain et forme un du...
4,NPA,oR4BqaFc68M,attends c'est quoi la chanson de la Moula c'es...
...,...,...,...
2515,Manon Aubry,V8KJ6NwKYwI,et en duplex depuis la place de la République ...
2516,Manon Aubry,TH7GNaX5too,Manon auri sur X anciennement Twitter vous ave...
2517,Marion Maréchal,AzoILatiQBk,et bonsoir Marion maréchal merci beaucoup d'êt...
2518,Marion Maréchal,9wViutuvmUg,qui a une situation inédite puisque pour la pr...


# Remove arte from everywhere

In [22]:
for filename in ["data/tiktok/videos/news_videos_2022.jsonl", 
                 "data/tiktok/videos/news_videos_2022_NER.jsonl",
                 "data/tiktok/videos/news_videos_2024.jsonl", 
                 "data/tiktok/videos/news_videos_2024_NER.jsonl"]:
    df = pd.read_json(filename, lines=True)
    df = df[df['name_standard'] != '28 minutes']
    df.to_json(filename, lines=True, orient='records', force_ascii=False)

In [24]:
for filename in ["data/youtube/videos/news_videos_2022.jsonl", 
                 "data/youtube/videos/news_videos_2024.jsonl"]:
    df = pd.read_json(filename, lines=True)
    df = df[df['name_standard'] != '28 minutes']
    df.to_json(filename, lines=True, orient='records', force_ascii=False)

In [3]:
for filename in sorted(os.listdir("data/pairs_of_transcripts/")):
    if ".csv" not in filename:continue
    print(filename)
    df = pd.read_csv(f"data/pairs_of_transcripts/{filename}")

pairs_diff_channels_tt_party_all_nm_2022.csv
pairs_diff_channels_tt_party_all_nm_2024.csv
pairs_diff_channels_tt_party_all_nm_2024_w_d.csv


/var/folders/s8/86mtzm812zdfvh1cgg7lh2wh0000gp/T/ipykernel_3600/1687531650.py:4: DtypeWarning: Columns (0: videoId2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"data/pairs_of_transcripts/{filename}")


pairs_diff_channels_tt_pp_all_nm_2022.csv
pairs_diff_channels_tt_pp_all_nm_2024.csv
pairs_diff_channels_tt_pp_all_nm_2024_w_d.csv


pairs_diff_channels_yt_party_all_nm_2022.csv
pairs_diff_channels_yt_party_all_nm_2024.csv
pairs_diff_channels_yt_party_all_nm_2024_w_d.csv


pairs_diff_channels_yt_pp_all_nm_2022.csv
pairs_diff_channels_yt_pp_all_nm_2024.csv
pairs_diff_channels_yt_pp_all_nm_2024_w_d.csv


...

pairs_same_actor_yt_tt_pp_2022.csv
pairs_same_actor_yt_tt_pp_2024.csv
pairs_same_actor_yt_tt_pp_2024_w_d.csv
pairs_same_party_yt_tt_2022.csv
pairs_same_party_yt_tt_2024.csv


pairs_same_party_yt_tt_2024_w_d.csv


pairs_yt_same_channel_nm_2022.csv
pairs_yt_same_channel_nm_2024.csv


pairs_yt_same_channel_nm_2024_w_d.csv


pairs_yt_same_channel_pp_2022.csv
pairs_yt_same_channel_pp_2024.csv
pairs_yt_same_channel_pp_2024_w_d.csv


In [4]:
df

,videoId1,videoId2,fw.partialratio_cut_t1,fw.sortratio_cut_t1,fw.setratio_cut_t1,fw.partialratio_cut_t2,fw.sortratio_cut_t2,fw.setratio_cut_t2
0,ehl5oLwYIFY,R3MJzzZTv_s,48,15,56,47,54,55
1,ehl5oLwYIFY,iGPHfnp3Bjo,48,15,56,46,54,55
2,ehl5oLwYIFY,SXBVd2WoEOY,47,16,56,48,54,53
3,ehl5oLwYIFY,ip72lruRuP0,48,16,54,47,52,55
4,ehl5oLwYIFY,T7LQXzEcsXY,48,8,67,47,55,57
...,...,...,...,...,...,...,...,...
50877,Y3GShqBpwAE,PXgelYxOp94,48,1,67,49,12,34
50878,Y3GShqBpwAE,Zzft-Tvg0l8,48,6,52,48,12,34
50879,Y3GShqBpwAE,D0UChyMVP7k,46,1,13,45,11,16
50880,Y3GShqBpwAE,PFV1ZcizRqw,50,1,72,47,12,30
